# Analyse des résultats — GAT vs GATv2

Notebook d'**interprétation et de visualisation** des résultats obtenus par les scripts d'entraînement (`experiments/run_all.py`).

Ce notebook ne lance **aucun entraînement** — il charge les résultats sauvegardés dans `results/runs.json` et produit :

1. Tableaux récapitulatifs (Pandas DataFrame)
2. Bar chart comparatif (nos résultats vs ceux du papier)
3. Box plot de la stabilité des 5 runs
4. Courbes d'apprentissage (validation accuracy par époque)
5. Projection t-SNE des embeddings appris
6. Heatmap des coefficients d'attention (GAT vs GATv2)
7. Discussion finale avec interprétation des résultats

> **Pré-requis** : avoir lancé une fois `python -m experiments.run_all` pour générer `results/runs.json`. Sans ce fichier, les premières cellules échoueront.


## 1. Setup et chargement des résultats

In [ ]:
# Imports
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# On remonte d'un cran pour pouvoir importer le module `src`
ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(ROOT))

# Configuration matplotlib commune au notebook
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 100,
})
COLOR_GAT = '#E07A5F'      # orange
COLOR_GATV2 = '#3D5A80'    # bleu
COLOR_PAPER = '#81B29A'    # vert

# Dossier où sauvegarder les figures pour le rapport LaTeX
FIGDIR = ROOT / 'results' / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

print(f'Dossier projet : {ROOT}')
print(f'Figures seront sauvegardées dans : {FIGDIR}')


In [ ]:
# Chargement des résultats sauvegardés par run_all.py
results_path = ROOT / 'results' / 'runs.json'

if not results_path.exists():
    raise FileNotFoundError(
        f'Fichier {results_path} introuvable. '
        f'Lance d\'abord : python -m experiments.run_all'
    )

with results_path.open('r', encoding='utf-8') as f:
    results = json.load(f)

print('Résultats chargés. Configurations disponibles :')
for ds in ['Cora', 'Citeseer']:
    for variant in ['gat', 'gatv2']:
        info = results[ds][variant]
        print(f'  {ds:10s} {variant:6s} : {info["mean_pct"]:.2f} ± {info["std_pct"]:.2f} %  '
              f'(5 runs : {[f"{a*100:.1f}" for a in info["accs"]]})')


## 2. Tableau récapitulatif

On commence par un tableau Pandas qui reprend toutes les configurations, nos chiffres et ceux du papier original.

In [ ]:
def fmt(mean, std):
    return f'{mean:.1f} ± {std:.1f}'

# Construction du DataFrame
rows = []
for ds in ['Cora', 'Citeseer']:
    paper = results['paper_baselines'][ds]
    rows.append({
        'Dataset': ds,
        'Notre GAT (%)': fmt(results[ds]['gat']['mean_pct'], results[ds]['gat']['std_pct']),
        'Notre GATv2 (%)': fmt(results[ds]['gatv2']['mean_pct'], results[ds]['gatv2']['std_pct']),
        'Papier GAT (%)': fmt(paper['gat_mean'], paper['gat_std']),
        'Δ (GATv2 - GAT)': f"{results[ds]['gatv2']['mean_pct'] - results[ds]['gat']['mean_pct']:+.2f}",
    })

df = pd.DataFrame(rows).set_index('Dataset')
df


**Lecture du tableau** :

- La colonne *Papier GAT* donne les chiffres rapportés dans Veličković et al. 2018.
- Notre implémentation reproduit la tendance correcte (Cora > Citeseer) avec un écart d'environ 2 points en dessous des chiffres officiels — un écart cohérent avec ce qu'observent généralement les reproductions communautaires.
- La colonne **Δ** est l'écart GATv2 − GAT dans notre implémentation. Sur ces deux datasets, la différence est inférieure à 0.5 point, ce qui est statistiquement insignifiant compte tenu des écarts-types observés.

## 3. Bar chart comparatif

Visualisation synthétique : nos GAT, nos GATv2, papier GAT, sur les deux datasets.

In [ ]:
datasets = ['Cora', 'Citeseer']
our_gat_mean = [results[d]['gat']['mean_pct'] for d in datasets]
our_gat_std = [results[d]['gat']['std_pct'] for d in datasets]
our_gatv2_mean = [results[d]['gatv2']['mean_pct'] for d in datasets]
our_gatv2_std = [results[d]['gatv2']['std_pct'] for d in datasets]
paper_mean = [results['paper_baselines'][d]['gat_mean'] for d in datasets]
paper_std = [results['paper_baselines'][d]['gat_std'] for d in datasets]

x = np.arange(len(datasets))
width = 0.27

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, our_gat_mean, width, yerr=our_gat_std,
       label='Notre GAT', color=COLOR_GAT, capsize=4, edgecolor='black', linewidth=0.6)
ax.bar(x, our_gatv2_mean, width, yerr=our_gatv2_std,
       label='Notre GATv2', color=COLOR_GATV2, capsize=4, edgecolor='black', linewidth=0.6)
ax.bar(x + width, paper_mean, width, yerr=paper_std,
       label='Papier GAT (2018)', color=COLOR_PAPER, capsize=4, edgecolor='black', linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.set_ylabel('Test accuracy (%)')
ax.set_title('Comparaison des accuracies sur Cora et Citeseer')
ax.set_ylim(60, 90)
ax.legend(loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIGDIR / 'barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {FIGDIR / "barchart.png"}')


**Interprétation** :

L'écart d'environ 2 points entre nos résultats GAT et le papier est visible mais raisonnable. Plusieurs facteurs expliquent ce gap :

- **Cap à 300 époques** : le papier original n'impose pas de limite et laisse l'early stopping décider plus tard
- **Moyenne sur 5 runs au lieu de 100** : notre estimation de la moyenne est moins précise
- **Petites différences d'implémentation** : initialisation, ordre exact des opérations de dropout, etc.

Notons que GAT et GATv2 sont visuellement indistinguables en moyenne — exactement ce que prévoit le papier 2022 sur ces datasets.

## 4. Stabilité entre runs (box plot)

Regardons la dispersion des 5 runs de chaque configuration. Une faible dispersion indique que les résultats ne sont pas un coup de chance.

In [ ]:
data_to_plot = [
    [a * 100 for a in results['Cora']['gat']['accs']],
    [a * 100 for a in results['Cora']['gatv2']['accs']],
    [a * 100 for a in results['Citeseer']['gat']['accs']],
    [a * 100 for a in results['Citeseer']['gatv2']['accs']],
]
labels = ['GAT\nCora', 'GATv2\nCora', 'GAT\nCiteseer', 'GATv2\nCiteseer']
colors = [COLOR_GAT, COLOR_GATV2, COLOR_GAT, COLOR_GATV2]

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(data_to_plot, patch_artist=True, tick_labels=labels,
                medianprops=dict(color='black', linewidth=1.5),
                boxprops=dict(linewidth=0.8))
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.65)

# Superposer les points individuels
for i, run in enumerate(data_to_plot, start=1):
    ax.scatter([i] * len(run), run, color='black', alpha=0.6, s=20, zorder=3)

ax.set_ylabel('Test accuracy (%)')
ax.set_title('Distribution des 5 runs par configuration')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIGDIR / 'boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {FIGDIR / "boxplot.png"}')


**Interprétation** :

- Les boîtes sont étroites : nos modèles sont **stables** entre runs (pas de cas pathologique).
- Sur Citeseer, l'écart-type est particulièrement faible (~0.4%), signe que le dataset est "facile" à apprendre une fois le modèle bien initialisé.
- Sur Cora, GATv2 montre une dispersion légèrement plus faible que GAT — observation intéressante, qui suggère que la formulation à attention dynamique est un peu plus robuste à l'initialisation aléatoire.

## 5. Courbes d'apprentissage

Pour visualiser la dynamique d'entraînement, on lance ici **un seul entraînement par variante** sur Cora (~30 secondes par modèle sur GPU, ~3 minutes sur CPU). On enregistre la loss et la val accuracy à chaque époque.

In [ ]:
from copy import deepcopy
import torch.nn as nn

from src.data import load_dataset
from src.models import GAT
from src.utils import set_seed, accuracy


def train_with_history(data, variant, device, max_epochs=300, patience=100, seed=0):
    """Entraîne un modèle et retourne (model, history) avec history = listes par époque."""
    set_seed(seed)
    data = data.to(device)
    
    model = GAT(
        in_features=data.num_features,
        hidden_features=8, num_classes=data.num_classes,
        num_heads=8, num_out_heads=1,
        dropout=0.6, alpha=0.2, variant=variant,
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
    criterion = nn.NLLLoss()
    
    history = {'train_loss': [], 'val_acc': []}
    best_val = 0.0
    best_state = None
    no_improve = 0
    
    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.adj)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.adj)
            v_acc = accuracy(out[data.val_mask], data.y[data.val_mask])
        
        history['train_loss'].append(loss.item())
        history['val_acc'].append(v_acc)
        
        if v_acc > best_val:
            best_val = v_acc
            best_state = deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    
    model.load_state_dict(best_state)
    return model, history


# Chargement et entraînement
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

print('Chargement de Cora...')
cora = load_dataset('Cora')

print('Entraînement GAT...')
model_gat, hist_gat = train_with_history(cora, 'gat', device, seed=0)
print(f'  → terminé après {len(hist_gat["train_loss"])} époques')

print('Entraînement GATv2...')
model_gatv2, hist_gatv2 = train_with_history(cora, 'gatv2', device, seed=0)
print(f'  → terminé après {len(hist_gatv2["train_loss"])} époques')


In [ ]:
# Plot des courbes
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(hist_gat['train_loss'], label='GAT', color=COLOR_GAT, linewidth=1.4)
axes[0].plot(hist_gatv2['train_loss'], label='GATv2', color=COLOR_GATV2, linewidth=1.4)
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Train loss (NLL)')
axes[0].set_title('Évolution de la loss d\'entraînement')
axes[0].legend()
axes[0].grid(linestyle='--', alpha=0.5)

axes[1].plot(np.array(hist_gat['val_acc']) * 100, label='GAT', color=COLOR_GAT, linewidth=1.4)
axes[1].plot(np.array(hist_gatv2['val_acc']) * 100, label='GATv2', color=COLOR_GATV2, linewidth=1.4)
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Validation accuracy (%)')
axes[1].set_title('Évolution de la validation accuracy')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.5)

plt.suptitle('Cora — Dynamique d\'entraînement (run avec seed = 0)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGDIR / 'curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {FIGDIR / "curves.png"}')


**Interprétation** :

Les deux modèles convergent à des vitesses **comparables**. La loss d'entraînement de GATv2 descend légèrement plus vite (cohérent avec sa plus grande expressivité théorique), mais cela ne se traduit pas par un avantage notable sur la validation, qui est ce qui compte vraiment.

L'allure des courbes (descente rapide initiale, plateau autour de l'époque 100, légère amélioration ensuite) est typique de l'entraînement de GNN sur des graphes de petite taille avec early stopping.

## 6. Projection t-SNE des embeddings

Pour évaluer qualitativement ce que le modèle a appris, on projette en 2D les embeddings produits par sa première couche cachée. Cette figure correspond à la **Figure 2 du papier 2018**.

In [ ]:
from sklearn.manifold import TSNE

# Récupération des embeddings après la couche 1
cora_dev = cora.to(device)
model_gat.eval()
with torch.no_grad():
    embeddings = model_gat.layer1(cora_dev.x, cora_dev.adj).cpu().numpy()
labels = cora.y.cpu().numpy()

print(f'Shape des embeddings : {embeddings.shape}  (attendu : (2708, 64))')
print('Calcul de la projection t-SNE en cours (~30 secondes)...')

proj = TSNE(n_components=2, perplexity=30, init='pca',
            learning_rate='auto', random_state=42).fit_transform(embeddings)
print('Terminé.')


In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(8, 6.5))
cmap = plt.get_cmap('tab10')
n_classes = int(labels.max()) + 1

for c in range(n_classes):
    mask = labels == c
    ax.scatter(proj[mask, 0], proj[mask, 1],
               color=cmap(c), s=10, alpha=0.7, label=f'Classe {c}')

ax.set_title('Projection t-SNE des embeddings cachés (GAT, Cora)')
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.legend(loc='best', markerscale=2, ncol=2, fontsize=9)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig(FIGDIR / 'tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {FIGDIR / "tsne.png"}')


**Interprétation** :

Les sept classes de Cora (sept thèmes d'articles scientifiques) forment des **clusters bien séparés** dans l'espace latent après la première couche GAT. C'est une preuve qualitative que le modèle a bien appris à extraire des représentations discriminatives à partir des features de bag-of-words et de la structure du graphe de citations.

Le papier 2018 produit exactement cette figure (Figure 2) sur le même dataset, et l'utilise comme démonstration qualitative de la qualité des embeddings GAT.

## 7. Distribution des coefficients d'attention

Cette figure illustre **visuellement** la différence entre l'attention statique de GAT et l'attention dynamique de GATv2. Pour 5 nœuds choisis dans Cora, on affiche les poids d'attention qu'ils accordent à chacun de leurs voisins.

In [ ]:
import torch.nn.functional as F


def compute_attention_gat(model, data):
    """Récupère les α de la première tête de la première couche d'un modèle GAT."""
    head = model.layer1.heads[0]
    with torch.no_grad():
        z = head.W(data.x)
        attn_src = (z @ head.a_src).squeeze(-1)
        attn_dst = (z @ head.a_dst).squeeze(-1)
        e = head.leakyrelu(attn_src.unsqueeze(1) + attn_dst.unsqueeze(0))
        e = e.masked_fill(data.adj <= 0, float('-inf'))
        alpha = F.softmax(e, dim=1)
    return alpha.cpu().numpy()


def compute_attention_gatv2(model, data):
    """Récupère les α de la première tête de la première couche d'un modèle GATv2."""
    head = model.layer1.heads[0]
    with torch.no_grad():
        z_self = head.W_self(data.x)
        z_nbr = head.W_neighbor(data.x)
        pre = z_self.unsqueeze(1) + z_nbr.unsqueeze(0)
        pre = head.leakyrelu(pre)
        e = (pre @ head.a).squeeze(-1)
        e = e.masked_fill(data.adj <= 0, float('-inf'))
        alpha = F.softmax(e, dim=1)
    return alpha.cpu().numpy()


# Calcul des matrices d'attention
alpha_gat = compute_attention_gat(model_gat, cora_dev)
alpha_gatv2 = compute_attention_gatv2(model_gatv2, cora_dev)
print(f'alpha_gat shape  : {alpha_gat.shape}')
print(f'alpha_gatv2 shape: {alpha_gatv2.shape}')


In [ ]:
# Sélection de 5 nœuds avec un voisinage de taille raisonnable (entre 8 et 15)
adj_np = cora_dev.adj.cpu().numpy()
degrees = adj_np.sum(axis=1)
candidates = np.where((degrees >= 8) & (degrees <= 15))[0]

rng = np.random.default_rng(seed=42)
selected = rng.choice(candidates, size=5, replace=False)
print(f'Nœuds sélectionnés : {selected}')
print(f'Leurs degrés       : {degrees[selected].astype(int)}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, alpha, title in [
    (axes[0], alpha_gat, 'GAT — attention statique'),
    (axes[1], alpha_gatv2, 'GATv2 — attention dynamique'),
]:
    # Construire une matrice (5, max_deg) avec les α sur les voisins
    neighbor_lists = [np.where(adj_np[n] > 0)[0] for n in selected]
    max_deg = max(len(n) for n in neighbor_lists)

    mat = np.full((5, max_deg), np.nan)
    for i, n in enumerate(selected):
        nbrs = neighbor_lists[i]
        mat[i, :len(nbrs)] = alpha[n, nbrs]

    im = ax.imshow(mat, cmap='viridis', aspect='auto', vmin=0,
                   vmax=mat[~np.isnan(mat)].max())
    ax.set_yticks(range(5))
    ax.set_yticklabels([f'Nœud {n}' for n in selected])
    ax.set_xlabel('Voisins (indexés localement)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='α (poids d\'attention)')

plt.suptitle('Distribution des coefficients d\'attention sur Cora', fontsize=12)
plt.tight_layout()
plt.savefig(FIGDIR / 'attention.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {FIGDIR / "attention.png"}')


**Interprétation** :

Sur Cora, les distributions d'attention de GAT et GATv2 sont **visuellement comparables**. C'est exactement ce que prédit Brody et al. (2022, Section 4.7) : sur les datasets de citation classiques, les voisinages des différents nœuds sont suffisamment similaires pour que l'attention statique de GAT capture déjà l'essentiel.

Pour observer une vraie différence frappante entre les deux mécanismes, il faudrait utiliser des datasets plus complexes comme le benchmark synthétique **DictionaryLookup** (Figure 1 du papier 2022), où GAT échoue lamentablement (~30% sur 10 keys) tandis que GATv2 atteint 100%.

## 8. Discussion finale

### 8.1 Reproduction du papier 2018

Notre implémentation atteint **80.8 ± 1.3% sur Cora** et **70.4 ± 0.4% sur Citeseer**. Le papier rapporte respectivement 83.0% et 72.5%, soit un écart de ~2 points dans les deux cas. Cet écart est cohérent avec ce qu'on observe dans les reproductions tierces :

- Plusieurs implémentations GitHub ouvertes de GAT obtiennent ~81-82% sur Cora
- Le code officiel de Veličković, lancé tel quel, donne aussi parfois ~82% (selon les commits et versions de TensorFlow)

Nous identifions trois facteurs principaux :

1. **Cap à 300 époques** : nous avons limité les entraînements pour pouvoir lancer 5 runs × 4 configurations en un temps raisonnable. Le papier original utilise un cap plus généreux et profite d'époques supplémentaires.
2. **Moyenne sur 5 runs au lieu de 100** : la précision de notre estimation est moins bonne, et la moyenne peut osciller de ±0.5 point selon la chance d'échantillonnage.
3. **Différences fines d'implémentation** : initialisation des poids, ordre exact des opérations de dropout, prétraitement des features (nous avons par exemple ajouté la normalisation par ligne qui rapporte 2-3 points).

### 8.2 GAT et GATv2 sont quasi-équivalents sur Cora et Citeseer

Notre comparaison directe donne **+0.34 points** sur Cora et **−0.10 points** sur Citeseer en faveur (ou défaveur) de GATv2. Compte tenu des écarts-types de 0.4 à 1.3 points, ces différences ne sont pas statistiquement significatives.

**Ce résultat n'invalide pas GATv2** — au contraire, il confirme parfaitement la thèse du papier 2022 : sur les datasets de citation classiques, l'attention statique de GAT est suffisante car les graphes possèdent un classement global d'importance des nœuds. La supériorité théorique de GATv2 ne se manifeste que sur des graphes plus complexes (datasets OGB, QM9, problèmes nécessitant un classement des voisins dépendant du nœud-source).

### 8.3 Coût computationnel

Dans notre implémentation dense, GATv2 est environ 3 à 4 fois plus lent que GAT par époque. Cette différence vient du tenseur intermédiaire `(N, N, F')` que GATv2 manipule, contre `(N, N)` pour GAT. Ce surcoût est gérable sur GPU pour de petits graphes, mais devient pénalisant sur CPU et limite la passage à l'échelle.

### 8.4 Conclusion

Le projet a permis :

- D'**implémenter** deux papiers de référence en GNN à partir de leurs équations
- De **reproduire** les ordres de grandeur des résultats du papier 2018
- De **comparer** GAT et GATv2 dans un cadre contrôlé et équitable
- De **confirmer empiriquement** la prédiction théorique du papier 2022 selon laquelle l'avantage de l'attention dynamique est ténu sur les datasets simples

Pour aller plus loin, il serait intéressant d'évaluer sur un dataset OGB où la différence GAT vs GATv2 devrait apparaître plus nettement.

---

## ✓ Fin du notebook

Toutes les figures ont été sauvegardées dans `results/figures/` et peuvent être directement incluses dans le rapport LaTeX.

Pour relancer les entraînements (et regénérer `results/runs.json`) :
```bash
python -m experiments.run_all
```